This is a Jupyter notebook for analyzing the Olist Brazilian E-commerce dataset.
Practicing in python with pandas and SQL to prepare for Elevate technical assessment. Working with Claude Code and visualizing in Windsurf.

<img src="olist_schema.png" width="700"/>

The Fictional Brief:
Elevate Portfolio Analytics — Olist Integration Project
Olist is a Brazilian e-commerce marketplace that connects small merchants to major retail channels. Your client, a PE firm, has acquired three regional merchant clusters operating on Olist's platform. Each cluster has its own sellers, customers, and order history. Your job is to unify and analyze the combined dataset to surface operational insights and flag data quality issues — exactly the kind of integration and BI work Elevate does across its roll-up companies.

Build Plan:

Phase 1 — Ingestion & Schema Mapping (Python/pandas)
Load all CSVs, inspect shapes and dtypes, identify primary and foreign keys across tables. Document your schema as a markdown cell with a simple entity map.

Phase 2 — Data Quality Audit (Python + DuckDB)
Use DuckDB to query across the CSVs with SQL — find nulls, duplicate keys, orphaned order items with no matching order, reviews with no matching customer. This is your integration QA layer.

Phase 3 — Core Joins & Feature Engineering (DuckDB SQL)
Build your master analytical table by joining orders → order_items → products → sellers → customers → reviews → payments. Add derived columns: delivery delay in days, a binary late/on-time flag, GMV per order.

Phase 4 — Seller Scorecard (pandas)
Aggregate to seller level: total GMV, order count, avg review score, late delivery rate. Rank and segment sellers into tiers.

Phase 5 — Visualizations (matplotlib/seaborn)
At minimum: a GMV concentration curve (Pareto), a delivery delay vs. review score scatter, and a regional heatmap by seller state.

Phase 6 — Written Summary
A markdown cell at the end stating your findings, which hypotheses held, which didn't, and what you'd recommend to the client.

## Phase 1 — Load, Inspect & Map Schema

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR = Path(".")

In [7]:
tables = {
    "orders":        pd.read_csv(DATA_DIR / "olist_orders_dataset.csv"),
    "order_items":   pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv"),
    "order_reviews": pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv"),
    "customers":     pd.read_csv(DATA_DIR / "olist_customers_dataset.csv"),
    "payments":      pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv"),
    "products":      pd.read_csv(DATA_DIR / "olist_products_dataset.csv"),
    "sellers":       pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv"),
    "geo":           pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv"),
    "category_xlat": pd.read_csv(DATA_DIR / "product_category_name_translation.csv"),
}
print("Loaded tables:", list(tables.keys()))

Loaded tables: ['orders', 'order_items', 'order_reviews', 'customers', 'payments', 'products', 'sellers', 'geo', 'category_xlat']


### Shape & Dtype Inspection

For each table: row/column counts, column dtypes, null counts, null %, and cardinality.

In [8]:
for name, df in tables.items():
    print(f"\n{'='*55}")
    print(f"TABLE: {name}  —  {df.shape[0]:,} rows × {df.shape[1]} cols")
    summary = pd.DataFrame({
        "dtype":    df.dtypes,
        "nulls":    df.isnull().sum(),
        "null_%":   (df.isnull().mean() * 100).round(2),
        "n_unique": df.nunique(),
    })
    display(summary)


TABLE: orders  —  99,441 rows × 8 cols


,dtype,nulls,null_%,n_unique
order_id,str,0,0.00,99441
customer_id,str,0,0.00,99441
order_status,str,0,0.00,8
order_purchase_timestamp,str,0,0.00,98875
order_approved_at,str,160,0.16,90733
order_delivered_carrier_date,str,1783,1.79,81018
order_delivered_customer_date,str,2965,2.98,95664
order_estimated_delivery_date,str,0,0.00,459



TABLE: order_items  —  112,650 rows × 7 cols


,dtype,nulls,null_%,n_unique
order_id,str,0,0.00,98666
order_item_id,int64,0,0.00,21
product_id,str,0,0.00,32951
seller_id,str,0,0.00,3095
shipping_limit_date,str,0,0.00,93318
price,float64,0,0.00,5968
freight_value,float64,0,0.00,6999



TABLE: order_reviews  —  99,224 rows × 7 cols


...

,dtype,nulls,null_%,n_unique
seller_id,str,0,0.00,3095
seller_zip_code_prefix,int64,0,0.00,2246
seller_city,str,0,0.00,611
seller_state,str,0,0.00,23



TABLE: geo  —  1,000,163 rows × 5 cols


,dtype,nulls,null_%,n_unique
geolocation_zip_code_prefix,int64,0,0.00,19015
geolocation_lat,float64,0,0.00,717363
geolocation_lng,float64,0,0.00,717615
geolocation_city,str,0,0.00,8011
geolocation_state,str,0,0.00,27



TABLE: category_xlat  —  71 rows × 2 cols


,dtype,nulls,null_%,n_unique
product_category_name,str,0,0.00,71
product_category_name_english,str,0,0.00,71


### Schema Relationship Map

```
orders (order_id PK, customer_id FK → customers.customer_id)
  ├─ order_items   (order_id FK, product_id FK → products, seller_id FK → sellers)
  ├─ payments      (order_id FK)
  └─ order_reviews (order_id FK)

customers    (customer_id PK, customer_zip_code_prefix → geo)
sellers      (seller_id PK,   seller_zip_code_prefix   → geo)
products     (product_id PK,  product_category_name FK → category_xlat)
geo          (geolocation_zip_code_prefix — NOT a unique PK; multiple lat/lng per zip)
category_xlat(product_category_name PK,  product_category_name_english)
```

**Cluster key** — `sellers.seller_state`:

| State | Cluster   |
|-------|-----------|
| SP    | Cluster A |
| RJ    | Cluster B |
| MG    | Cluster C |

**FK → PK reference table**

| Child table   | FK column                   | Parent table  | PK column                          |
|---------------|-----------------------------|---------------|------------------------------------|
| orders        | customer_id                 | customers     | customer_id                        |
| order_items   | order_id                    | orders        | order_id                           |
| order_items   | seller_id                   | sellers       | seller_id                          |
| order_items   | product_id                  | products      | product_id                         |
| payments      | order_id                    | orders        | order_id                           |
| order_reviews | order_id                    | orders        | order_id                           |
| products      | product_category_name       | category_xlat | product_category_name              |
| customers     | customer_zip_code_prefix    | geo           | geolocation_zip_code_prefix (non-unique) |
| sellers       | seller_zip_code_prefix      | geo           | geolocation_zip_code_prefix (non-unique) |

### Referential Integrity Checks

Verify FK → PK joins across all core relationships. Large orphan counts flag data quality gaps worth investigating in later phases.

In [9]:
checks = {
    "order_items.order_id   → orders":    (tables["order_items"]["order_id"],
                                            tables["orders"]["order_id"]),
    "order_items.seller_id  → sellers":   (tables["order_items"]["seller_id"],
                                            tables["sellers"]["seller_id"]),
    "order_items.product_id → products":  (tables["order_items"]["product_id"],
                                            tables["products"]["product_id"]),
    "orders.customer_id     → customers": (tables["orders"]["customer_id"],
                                            tables["customers"]["customer_id"]),
    "payments.order_id      → orders":    (tables["payments"]["order_id"],
                                            tables["orders"]["order_id"]),
    "reviews.order_id       → orders":    (tables["order_reviews"]["order_id"],
                                            tables["orders"]["order_id"]),
    "products.category      → xlat":      (tables["products"]["product_category_name"].dropna(),
                                            tables["category_xlat"]["product_category_name"]),
}

print(f"{'Relationship':<45}  {'Orphans':>10}  {'Total FK':>10}  {'Orphan %':>9}")
print("-" * 80)
for label, (fk_col, pk_col) in checks.items():
    orphans = (~fk_col.isin(pk_col)).sum()
    total   = len(fk_col)
    pct     = orphans / total * 100 if total else 0
    flag    = " ⚠️" if orphans > 0 else ""
    print(f"{label:<45}  {orphans:>10,}  {total:>10,}  {pct:>8.2f}%{flag}")

Relationship                                      Orphans    Total FK   Orphan %
--------------------------------------------------------------------------------
order_items.order_id   → orders                         0     112,650      0.00%
order_items.seller_id  → sellers                        0     112,650      0.00%
order_items.product_id → products                       0     112,650      0.00%
orders.customer_id     → customers                      0      99,441      0.00%
payments.order_id      → orders                         0     103,886      0.00%
reviews.order_id       → orders                         0      99,224      0.00%
products.category      → xlat                          13      32,341      0.04% ⚠️


### Cluster Assignment

Tag each seller by state → cluster label. This is the segmentation key for all downstream analysis.

In [10]:
CLUSTER_MAP = {"SP": "Cluster A", "RJ": "Cluster B", "MG": "Cluster C"}

sellers = tables["sellers"].copy()
sellers["cluster"] = sellers["seller_state"].map(CLUSTER_MAP).fillna("Other")
tables["sellers"] = sellers  # update shared dict for downstream use

cluster_counts = sellers["cluster"].value_counts().rename_axis("cluster").reset_index(name="seller_count")
cluster_counts["pct"] = (cluster_counts["seller_count"] / cluster_counts["seller_count"].sum() * 100).round(1)
display(cluster_counts)

,cluster,seller_count,pct
0,Cluster A,1849,59.70
1,Other,831,26.80
2,Cluster C,244,7.90
3,Cluster B,171,5.50


## Phase 2 — Data Quality Audit (DuckDB)

We register the pandas DataFrames from Phase 1 as DuckDB views, then run SQL checks against them.
DuckDB is an in-process OLAP engine — no server, no setup, works directly on in-memory DataFrames.

**Checks in this phase:**
1. DuckDB setup & view registration
2. Duplicate primary key detection
3. NULL audit on key columns
4. Orphaned record checks (LEFT JOIN + WHERE IS NULL pattern)
5. Payment anomalies
6. QA summary report (PASS / WARN)

In [11]:
import duckdb

con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)

registered = con.execute("SHOW TABLES").df()["name"].tolist()
print("DuckDB views registered:", registered)

DuckDB views registered: ['category_xlat', 'customers', 'geo', 'order_items', 'order_reviews', 'orders', 'payments', 'products', 'sellers']


### 1. Duplicate Primary Key Detection

Duplicate PKs would silently fan-out rows in every downstream join, inflating counts and GMV.
Pattern: `COUNT(*) - COUNT(DISTINCT pk_col)` — any non-zero result is a problem.

In [12]:
pk_checks = [
    ("orders",    "order_id"),
    ("customers", "customer_id"),
    ("sellers",   "seller_id"),
    ("products",  "product_id"),
    ("category_xlat", "product_category_name"),
]

rows = []
for table, pk_col in pk_checks:
    result = con.execute(f"""
        SELECT
            '{table}'  AS table_name,
            '{pk_col}' AS pk_column,
            COUNT(*)                        AS total_rows,
            COUNT(DISTINCT {pk_col})        AS unique_keys,
            COUNT(*) - COUNT(DISTINCT {pk_col}) AS duplicates
        FROM {table}
    """).df()
    rows.append(result)

pk_summary = pd.concat(rows, ignore_index=True)
display(pk_summary)

,table_name,pk_column,total_rows,unique_keys,duplicates
0,orders,order_id,99441,99441,0
1,customers,customer_id,99441,99441,0
2,sellers,seller_id,3095,3095,0
3,products,product_id,32951,32951,0
4,category_xlat,product_category_name,71,71,0


### 2. NULL Audit on Key Columns

NULLs in join keys silently drop rows during joins. NULLs in metric columns silently undercount revenue or delivery stats.
We target columns that are either join keys or will be used in downstream calculations.

In [13]:
null_targets = {
    "orders":        ["order_approved_at", "order_delivered_carrier_date",
                      "order_delivered_customer_date"],
    "order_items":   ["price", "freight_value"],
    "products":      ["product_category_name", "product_weight_g"],
    "order_reviews": ["review_comment_title", "review_comment_message"],
}

null_rows = []
for table, cols in null_targets.items():
    col_exprs = ",\n        ".join(
        f"COUNT(*) - COUNT({c}) AS {c}" for c in cols
    )
    total_q = f"SELECT COUNT(*) AS n FROM {table}"
    total   = con.execute(total_q).fetchone()[0]

    result  = con.execute(f"SELECT {col_exprs} FROM {table}").df()
    for col in cols:
        null_count = int(result[col].iloc[0])
        null_rows.append({
            "table":      table,
            "column":     col,
            "null_count": null_count,
            "null_pct":   round(null_count / total * 100, 2),
        })

null_summary = pd.DataFrame(null_rows)
display(null_summary[null_summary["null_count"] > 0].reset_index(drop=True))

,table,column,null_count,null_pct
0,orders,order_approved_at,160,0.16
1,orders,order_delivered_carrier_date,1783,1.79
2,orders,order_delivered_customer_date,2965,2.98
3,products,product_category_name,610,1.85
4,products,product_weight_g,2,0.01
5,order_reviews,review_comment_title,87656,88.34
6,order_reviews,review_comment_message,58247,58.70


### 3. Orphaned Record Checks

**LEFT JOIN + WHERE IS NULL** is the canonical SQL orphan-detection idiom.
Each query surfaces records in a child table with no matching parent — a sign of bad data ingestion
or integration gaps between clusters.

In [14]:
orphan_queries = {
    "order_items with no parent order": """
        SELECT COUNT(*) AS n
        FROM order_items oi
        LEFT JOIN orders o ON oi.order_id = o.order_id
        WHERE o.order_id IS NULL
    """,
    "reviews with no matching order": """
        SELECT COUNT(*) AS n
        FROM order_reviews r
        LEFT JOIN orders o ON r.order_id = o.order_id
        WHERE o.order_id IS NULL
    """,
    "reviews with no reachable customer": """
        SELECT COUNT(*) AS n
        FROM order_reviews r
        LEFT JOIN orders o ON r.order_id = o.order_id
        LEFT JOIN customers c ON o.customer_id = c.customer_id
        WHERE c.customer_id IS NULL
    """,
    "orders with no items (ghost/cancelled)": """
        SELECT COUNT(*) AS n
        FROM orders o
        LEFT JOIN order_items oi ON o.order_id = oi.order_id
        WHERE oi.order_id IS NULL
    """,
}

orphan_results = {}
for label, sql in orphan_queries.items():
    count = con.execute(sql).fetchone()[0]
    orphan_results[label] = count
    flag = "⚠️ " if count > 0 else "✓  "
    print(f"  {flag}{label:<45}  {count:,}")

  ✓  order_items with no parent order               0
  ✓  reviews with no matching order                 0
  ✓  reviews with no reachable customer             0
  ⚠️ orders with no items (ghost/cancelled)         775


### 4. Payment Anomaly Checks

Multi-installment payments (parcelamento) are normal in Brazil — one order can have several payment rows.
Anomalies to flag: orders with no payment record, and orders where total payment value = 0.

In [15]:
orders_no_payment = con.execute("""
    SELECT COUNT(*) AS n
    FROM orders o
    LEFT JOIN payments p ON o.order_id = p.order_id
    WHERE p.order_id IS NULL
""").fetchone()[0]

zero_value_orders = con.execute("""
    SELECT COUNT(*) AS n FROM (
        SELECT order_id
        FROM payments
        GROUP BY order_id
        HAVING SUM(payment_value) = 0
    )
""").fetchone()[0]

payment_type_dist = con.execute("""
    SELECT payment_type,
           COUNT(DISTINCT order_id) AS orders,
           ROUND(COUNT(DISTINCT order_id) * 100.0 / SUM(COUNT(DISTINCT order_id)) OVER (), 1) AS pct
    FROM payments
    GROUP BY payment_type
    ORDER BY orders DESC
""").df()

print(f"  {'⚠️' if orders_no_payment > 0 else '✓ '} Orders with no payment record : {orders_no_payment:,}")
print(f"  {'⚠️' if zero_value_orders  > 0 else '✓ '} Orders with payment value = 0 : {zero_value_orders:,}")
print()
print("Payment type distribution:")
display(payment_type_dist)

  ⚠️ Orders with no payment record : 1
  ⚠️ Orders with payment value = 0 : 3

Payment type distribution:


,payment_type,orders,pct
0,credit_card,76505,75.20
1,boleto,19784,19.50
2,voucher,3866,3.80
3,debit_card,1528,1.50
4,not_defined,3,0.00


### 5. QA Summary Report

Aggregated PASS / WARN report — the stakeholder artifact for this phase.
Flag anything non-zero as WARN; the narrative below each result is what you'd say in a meeting.

In [16]:
qa_checks = []

# --- duplicate PKs ---
for _, row in pk_summary.iterrows():
    qa_checks.append({
        "check":  f"Duplicate PKs — {row['table_name']}.{row['pk_column']}",
        "result": "PASS" if row["duplicates"] == 0 else "WARN",
        "count":  int(row["duplicates"]),
    })

# --- nulls on key columns ---
for _, row in null_summary[null_summary["null_count"] > 0].iterrows():
    qa_checks.append({
        "check":  f"NULLs — {row['table']}.{row['column']}",
        "result": "WARN",
        "count":  int(row["null_count"]),
    })

# --- orphaned records ---
for label, count in orphan_results.items():
    qa_checks.append({
        "check":  label,
        "result": "WARN" if count > 0 else "PASS",
        "count":  count,
    })

# --- payment anomalies ---
qa_checks.append({"check": "Orders with no payment record", "result": "WARN" if orders_no_payment > 0 else "PASS", "count": orders_no_payment})
qa_checks.append({"check": "Orders with payment_value = 0",  "result": "WARN" if zero_value_orders  > 0 else "PASS", "count": zero_value_orders})

# --- print report ---
LINE = "═" * 72
print(f"╔{LINE}╗")
print(f"║{'  DATA QUALITY AUDIT — PHASE 2 SUMMARY':^72}║")
print(f"╠{LINE}╣")
print(f"║  {'Check':<48}  {'Result':>6}  {'Count':>8}  ║")
print(f"╠{LINE}╣")
for c in qa_checks:
    tag   = "⚠️  WARN" if c["result"] == "WARN" else "✓  PASS"
    count = f"{c['count']:,}" if c["count"] > 0 else "—"
    print(f"║  {c['check']:<48}  {tag:>8}  {count:>8}  ║")
print(f"╚{LINE}╝")

n_warn = sum(1 for c in qa_checks if c["result"] == "WARN")
print(f"\n  {n_warn} warning(s) flagged. Address before building Phase 3 metrics.")

╔════════════════════════════════════════════════════════════════════════╗
║                   DATA QUALITY AUDIT — PHASE 2 SUMMARY                 ║
╠════════════════════════════════════════════════════════════════════════╣
║  Check                                             Result     Count  ║
╠════════════════════════════════════════════════════════════════════════╣
║  Duplicate PKs — orders.order_id                    ✓  PASS         —  ║
║  Duplicate PKs — customers.customer_id              ✓  PASS         —  ║
║  Duplicate PKs — sellers.seller_id                  ✓  PASS         —  ║
║  Duplicate PKs — products.product_id                ✓  PASS         —  ║
║  Duplicate PKs — category_xlat.product_category_name   ✓  PASS         —  ║
║  NULLs — orders.order_approved_at                  ⚠️  WARN       160  ║
║  NULLs — orders.order_delivered_carrier_date       ⚠️  WARN     1,783  ║
║  NULLs — orders.order_delivered_customer_date      ⚠️  WARN     2,965  ║
║  NULLs — products.prod

# Data Quality Audit - Review Comments (10 Warnings)

# Expected Behavior:
1. order_reviews.review_comment_title (87,656 nulls) and order_reviews.review_comment_message (58,247 nulls) - most customers leave a score rating without writing anything. Not missing data, valid empty responses.

3. orders.order_approved_at (160 nulls) - orders never approved, likely cancelled at checkout. Not relevant to our delivery / GMV analysis 

# Handle with a filter in Phase 3

4. orders.order_delivered_carrier_date (1,783 nulls) - orders not yet delivered by carrier and orders.order_delivered_customer_date (2,965 nulls) - orders never delivered, still in transit, or cancelled. Filter these out in phase 3, "exclude rows where order_delivered_carrier_date is null"

6. orders with no items / ghost orders (775) - these are orders that don't have any items associated with them. We should filter these out in phase 3.

7. products.product_category_name (610 nulls) — uncategorized products. Flag in your written summary as a data quality gap in the source system, use "Unknown" as a fill value.
8. products.product_weight_g (2 nulls) — negligible, won't affect anything.
10. Orders with no payment record (1) and Orders with payment_value = 0 (3) — four edge case orders total. Exclude from cost and GMV analysis, note them in your summary.




## Phase 2.5 — Data Quality Decisions & Analytical View Definitions

The following decisions resolve all Phase 2 warnings. Each decision is scoped to the analysis
it affects — no data is deleted from the raw `tables` dict, only filtered views are created.

| # | Issue | Decision | Scope |
|---|-------|----------|-------|
| 1 | `orders.order_approved_at` — 160 NULLs | **Exclude from all analysis.** Unapproved orders represent payment failures or fraud holds; no GMV was realised and no fulfillment occurred. | All downstream |
| 2 | `orders.order_delivered_customer_date` — 2,965 NULLs | **Exclude from delivery delay calculations only.** Undelivered/in-transit orders cannot produce a valid delay figure; they are retained in order volume counts. | Delivery metrics |
| 3 | `orders with no items` — 775 ghost orders | **Exclude from GMV and seller analysis.** No items means no revenue and no seller attribution. | GMV, seller scorecard |
| 4 | `products.product_category_name` — 610 NULLs | **Fill with `'Unknown'`.** Retains all products in category-level breakdowns; flagged as a source-system gap in the written summary. | Category analysis |
| 5 | `order_reviews` — 814 duplicate `review_id`s | **Deduplicate to one review per `order_id`, keeping the highest `review_score`.** Conservative: does not penalise sellers for duplicated records. | Review scores |
| 6 | Orders with no payment record (1) or `payment_value = 0` (3) | **Exclude from cost and GMV analysis.** Four edge-case orders with no economic signal. | GMV, cost analysis |
| 7 | Sellers outside SP / RJ / MG — 831 sellers | **Retain as explicit "Other" cluster** (already tagged in Phase 1). Included in seller scorecard as a fourth segment; excluded only if a cluster-restricted view is needed. | Seller scorecard |

> **Note on `order_approved_at` vs. conversion funnel:** If a future phase builds a
> placed → approved → shipped → delivered funnel, the 160 unapproved orders should be
> reintroduced to that specific view to show top-of-funnel drop-off.

In [ ]:
# ── Decision 4: fill null product categories ──────────────────────────────
tables["products"]["product_category_name"] = (
    tables["products"]["product_category_name"].fillna("Unknown")
)

# ── Decision 5: deduplicate reviews — one per order_id, highest score ─────
tables["order_reviews"] = (
    tables["order_reviews"]
    .sort_values("review_score", ascending=False)
    .drop_duplicates(subset="order_id", keep="first")
    .reset_index(drop=True)
)

# Re-register the two updated DataFrames so DuckDB views reflect the fixes
for name in ["products", "order_reviews"]:
    con.register(name, tables[name])

print(f"products.product_category_name nulls remaining : "
      f"{tables['products']['product_category_name'].isnull().sum()}")
print(f"order_reviews rows after dedup                 : "
      f"{len(tables['order_reviews']):,}  "
      f"(was 99,224 — {99_224 - len(tables['order_reviews']):,} duplicates removed)")

In [ ]:
# ── Decision 1: base_orders — approved orders only (excludes 160 unapproved) ──
con.execute("""
    CREATE OR REPLACE VIEW base_orders AS
    SELECT * FROM orders
    WHERE order_approved_at IS NOT NULL
""")

# ── Decision 2: orders_for_delivery — must also have a delivery date ───────────
con.execute("""
    CREATE OR REPLACE VIEW orders_for_delivery AS
    SELECT * FROM base_orders
    WHERE order_delivered_customer_date IS NOT NULL
""")

# ── Decisions 3 & 6: orders_for_gmv — base + has items + valid payment ────────
con.execute("""
    CREATE OR REPLACE VIEW valid_payment_ids AS
    SELECT order_id
    FROM payments
    GROUP BY order_id
    HAVING SUM(payment_value) > 0
""")

con.execute("""
    CREATE OR REPLACE VIEW orders_for_gmv AS
    SELECT o.* FROM base_orders o
    WHERE o.order_id IN (SELECT DISTINCT order_id FROM order_items)
      AND o.order_id IN (SELECT order_id FROM valid_payment_ids)
""")

print("Analytical views created:")
for v in ["base_orders", "orders_for_delivery", "valid_payment_ids", "orders_for_gmv"]:
    n = con.execute(f"SELECT COUNT(*) FROM {v}").fetchone()[0]
    print(f"  {v:<25}  {n:,} rows")

In [ ]:
raw_orders = len(tables["orders"])

summary_rows = [
    ("Raw orders",                     raw_orders,  "—"),
    ("base_orders (approved)",
     con.execute("SELECT COUNT(*) FROM base_orders").fetchone()[0],
     "excl. 160 unapproved (fraud / payment failure)"),
    ("orders_for_delivery",
     con.execute("SELECT COUNT(*) FROM orders_for_delivery").fetchone()[0],
     "further excl. orders with no delivery date"),
    ("orders_for_gmv",
     con.execute("SELECT COUNT(*) FROM orders_for_gmv").fetchone()[0],
     "further excl. ghost orders + zero-payment orders"),
]

print(f"\n{'View':<30}  {'Rows':>8}  {'Excluded':>8}  Note")
print("─" * 80)
for label, n, note in summary_rows:
    excluded = raw_orders - n if label != "Raw orders" else 0
    excl_str = f"{excluded:,}" if excluded else "—"
    print(f"{label:<30}  {n:>8,}  {excl_str:>8}  {note}")